# Scenario impact analysis

Stress a **`PortfolioSpec`** against a **`MarketContext`** snapshot, then let Rust decompose the resulting P&L by risk factor.

The canonical path is **`portfolio.attribute_portfolio_pnl`**: give it the **T₀** market and the **T₁** (stressed) market and it returns a typed **`PortfolioAttribution`** with `rates_curves_pnl`, `credit_curves_pnl`, `fx_pnl`, `fx_translation_pnl` and `cross_factor_pnl`, plus a **`reconciliation_check`** that proves the buckets add back to the total.

At the end we run the naive "apply each shock on its own and difference the PVs" decomposition **as a cross-check**, to show why it is *not* the same thing as the engine's answer.

## Concept

The workflow has four steps, each one binding call:

1. **Value the base book** — `value_portfolio(spec_json, market_json)` returns `PortfolioValuation` JSON. Wrap it with `PortfolioValuation.from_json(...)` and read **`.total_value`** rather than digging into `total_base_ccy["amount"]` by hand.
2. **Author the shocks** — build operations with the typed **`OperationSpec`** builders (`curve_parallel_bp`, `market_fx_pct`, …) instead of hand-writing raw JSON dicts, then wrap each factor in its own `ScenarioSpec` via `build_scenario_spec`.
3. **Combine them** — **`compose_scenarios`** merges the per-factor specs into a single spec using the engine's own `priority` ordering. Concatenating `operations` arrays in Python bypasses that ordering logic.
4. **Move the market and attribute** — **`apply_scenario_to_market`** produces the T₁ `MarketContext`; **`attribute_portfolio_pnl(portfolio, market_t0, market_t1, as_of_t0, as_of_t1, method)`** decomposes T₁ − T₀ into typed factor buckets.

**Factor mapping** (`AttributionFactor` → `MarketContext`):

| Bucket | Driven by |
| --- | --- |
| `rates_curves_pnl` | discount + forward curves (`USD-OIS`, `EUR-OIS` here) |
| `credit_curves_pnl` | hazard curves (`CORP-HAZARD`, shocked through `par_cds`) |
| `fx_pnl` | revaluation of FX-sensitive instruments (FX forwards/swaps) |
| `fx_translation_pnl` | translating foreign-currency position PVs into base currency |
| `cross_factor_pnl` | the interaction term the naive sum-of-singles misses |

A plain foreign-currency deposit carries **no** FX-instrument risk, so its EUR/USD sensitivity lands in **`fx_translation_pnl`**, not `fx_pnl`. Watch for that below.

In [1]:
import json

import sys
sys.path.insert(0, "..")

from _shared import DEMO_AS_OF, build_demo_market
from finstack_quant.portfolio import PortfolioValuation, value_portfolio

# Shared cross-asset market: USD-OIS, EUR-OIS, USD-SOFR-3M, CORP-HAZARD, EUR/USD = 1.08.
AS_OF = DEMO_AS_OF.isoformat()
market = build_demo_market()
market_json = market.to_json()
print(f"Market ready ({AS_OF}): USD-OIS, EUR-OIS, CORP-HAZARD, EUR/USD=1.08")

portfolio = json.dumps(
    {
        "id": "attrib-demo",
        "as_of": AS_OF,
        "base_ccy": "USD",
        "entities": {"E1": {"id": "E1"}},
        "positions": [
            {
                "position_id": "P-USD",
                "entity_id": "E1",
                "instrument_id": "DEP-USD",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-USD",
                        "notional": {"amount": 1_000_000.0, "currency": "USD"},
                        "start_date": AS_OF,
                        "maturity": "2025-10-15",
                        "day_count": "Act360",
                        "quote_rate": 0.052,
                        "discount_curve_id": "USD-OIS",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            },
            {
                "position_id": "P-EUR",
                "entity_id": "E1",
                "instrument_id": "DEP-EUR",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-EUR",
                        "notional": {"amount": 400_000.0, "currency": "EUR"},
                        "start_date": AS_OF,
                        "maturity": "2025-10-15",
                        "day_count": "Act360",
                        "quote_rate": 0.038,
                        "discount_curve_id": "EUR-OIS",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            },
            {
                "position_id": "P-CDS",
                "entity_id": "E1",
                "instrument_id": "CDS-1",
                "instrument_spec": {
                    "type": "credit_default_swap",
                    "spec": {
                        "id": "CDS-1",
                        "notional": {"amount": 1_500_000.0, "currency": "USD"},
                        "side": "pay",
                        "convention": "isda_na",
                        "premium": {
                            "start": AS_OF,
                            "end": "2030-01-15",
                            "frequency": {"count": 3, "unit": "months"},
                            "stub": "ShortFront",
                            "bdc": "following",
                            "calendar_id": None,
                            "day_count": "Act360",
                            "spread_bp": 110.0,
                            "discount_curve_id": "USD-OIS",
                        },
                        "protection": {
                            "credit_curve_id": "CORP-HAZARD",
                            "recovery_rate": 0.4,
                            "settlement_delay": 3,
                        },
                        "pricing_overrides": {},
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            },
        ],
    }
)
print("Portfolio positions: USD deposit, EUR deposit, USD CDS on CORP-HAZARD")

# `PortfolioValuation.total_value` is the typed accessor for `total_base_ccy`.
base_valuation = PortfolioValuation.from_json(value_portfolio(portfolio, market_json))
base_total = base_valuation.total_value
print(f"Base total ({base_valuation.base_ccy}): {base_total:,.2f}")


Market ready (2025-01-15): USD-OIS, EUR-OIS, CORP-HAZARD, EUR/USD=1.08
Portfolio positions: USD deposit, EUR deposit, USD CDS on CORP-HAZARD


Base total (USD): 1,461,825.01


## Authoring the stress: typed operations, composed once

Each risk factor gets its own `ScenarioSpec` so we can also reuse it for the single-factor cross-check later. `compose_scenarios` then merges the three into the one spec that defines T₁.

`priority` controls merge ordering (lower runs earlier). FX is priced first, then rates, then credit — but because we apply the merged spec in a single call, ordering only matters where operations do not commute.

In [2]:
from finstack_quant.scenarios import (
    CurveKind,
    OperationSpec,
    apply_scenario_to_market,
    build_scenario_spec,
    compose_scenarios,
    validate_scenario_spec,
)


def operations_json(*ops: OperationSpec) -> str:
    """Serialize typed `OperationSpec` builders into the wire list `build_scenario_spec` expects."""
    return json.dumps([json.loads(op.to_json()) for op in ops])


rates_scn = build_scenario_spec(
    "rates",
    operations_json(
        OperationSpec.curve_parallel_bp(CurveKind.discount(), "USD-OIS", 45.0),
        OperationSpec.curve_parallel_bp(CurveKind.discount(), "EUR-OIS", 45.0),
    ),
    "Rates +45bp (USD-OIS, EUR-OIS)",
    "",
    priority=10,
)
credit_scn = build_scenario_spec(
    "credit",
    operations_json(
        OperationSpec.curve_parallel_bp(
            CurveKind.par_cds(), "CORP-HAZARD", 85.0, discount_curve_id="USD-OIS"
        ),
    ),
    "Credit +85bp par CDS",
    "",
    priority=20,
)
fx_scn = build_scenario_spec(
    "fx",
    operations_json(OperationSpec.market_fx_pct("EUR", "USD", -3.0)),
    "FX -3% EURUSD (EUR weakens)",
    "",
    priority=5,
)
print("Specs valid:", validate_scenario_spec(rates_scn), validate_scenario_spec(credit_scn), validate_scenario_spec(fx_scn))

# Engine-owned composition — never concatenate `operations` arrays in Python.
combined_scn = compose_scenarios(json.dumps([json.loads(s) for s in (rates_scn, credit_scn, fx_scn)]))
combined_obj = json.loads(combined_scn)
print(f"Composed scenario id={combined_obj['id']!r} operations={len(combined_obj['operations'])}")
for i, op in enumerate(combined_obj["operations"]):
    print(f"  [{i}] kind={op['kind']!r} ref={(op.get('curve_id') or op.get('base'))!r}")

# T1 market = base market with every composed operation applied.
applied = apply_scenario_to_market(combined_scn, market_json, AS_OF)
market_t1_json = applied["market_json"]
print("\noperations_applied:", applied["operations_applied"], "warnings:", applied["warnings"])

stressed_total = PortfolioValuation.from_json(value_portfolio(portfolio, market_t1_json)).total_value
print(f"Base total     (USD): {base_total:,.2f}")
print(f"Stressed total (USD): {stressed_total:,.2f}")
print(f"Total dP&L     (USD): {stressed_total - base_total:,.2f}")

Specs valid: True True True
Composed scenario id='fx+rates+credit' operations=4
  [0] kind='market_fx_pct' ref='EUR'
  [1] kind='curve_parallel_bp' ref='USD-OIS'
  [2] kind='curve_parallel_bp' ref='EUR-OIS'
  [3] kind='curve_parallel_bp' ref='CORP-HAZARD'



operations_applied: 4 warnings: []


Base total     (USD): 1,461,825.01
Stressed total (USD): 1,495,501.08
Total dP&L     (USD): 33,676.06


## Attributing the P&L: choosing a method

`attribute_portfolio_pnl` takes a `method` in the same serde shape as instrument attribution. Four are available:

| Method | Wire form | How it decomposes | When to use it |
| --- | --- | --- | --- |
| **Parallel** | `"Parallel"` | Full reprice; isolates each factor by restoring its **T₀** value while every other factor stays at **T₁**. Interactions land in `cross_factor_pnl`. | **Default for scenario impact** — exact repricing, explicit interaction term. |
| **Waterfall** | `{"Waterfall": ["RatesCurves", "CreditCurves", "Fx"]}` | Applies factors sequentially, each on top of the previous. Buckets sum to the total by construction, but depend on the order you pick. | Reporting that must be additive and where you can defend an ordering. |
| **MetricsBased** | `"MetricsBased"` | Linear approximation from pre-computed DV01 / CS01 / Theta. | Fast estimates; drops convexity, so poor for large shocks. |
| **Taylor** | `{"Taylor": {...}}` | Bump-and-reprice sensitivities at T₀ × observed market moves, optionally with second-order terms. | Explaining P&L in the language of the risk report. |

We use **`Parallel`** here: our shocks are large (+45bp, +85bp, −3%), so a linear method would misstate them, and the whole point of the notebook is to *see* the interaction term rather than allocate it away.

`as_of_t0 == as_of_t1` because a scenario is an instantaneous market move, not a passage of time — which is why `carry` comes back at zero.

In [3]:
from finstack_quant.portfolio import attribute_portfolio_pnl

attribution = attribute_portfolio_pnl(
    portfolio,
    market_json,      # T0
    market_t1_json,   # T1 = composed stress applied
    AS_OF,            # as_of_t0
    AS_OF,            # as_of_t1 — instantaneous shock, so no carry
    "Parallel",
)

buckets = [
    ("Carry", attribution.carry),
    ("Rates curves", attribution.rates_curves_pnl),
    ("Credit curves", attribution.credit_curves_pnl),
    ("FX (instruments)", attribution.fx_pnl),
    ("FX translation", attribution.fx_translation_pnl),
    ("Cross-factor", attribution.cross_factor_pnl),
    ("Residual", attribution.residual),
]
print("Rust factor decomposition (method=Parallel)")
for label, money in buckets:
    print(f"  {label:<18} {money.amount:>14,.2f} {money.currency.code}")
total = attribution.total_pnl
print(f"  {'TOTAL':<18} {total.amount:>14,.2f} {total.currency.code}")

# The engine proves its own arithmetic rather than asking Python to re-add the buckets.
check = attribution.reconciliation_check(1e-6)
print(
    f"\nreconciliation_check: reconciled={check['is_reconciled']} "
    f"residual={check['total_residual']:.3e} tolerance={check['tolerance']:g}"
)

# Independent tie-out: the attributed total must equal the repriced PV difference.
print(f"attribution total_pnl : {total.amount:,.2f}")
print(f"stressed PV - base PV : {stressed_total - base_total:,.2f}")

Rust factor decomposition (method=Parallel)
  Carry                        0.00 USD
  Rates curves            -5,622.67 USD
  Credit curves           51,777.19 USD
  FX (instruments)             0.00 USD
  FX translation         -13,034.49 USD
  Cross-factor               556.03 USD
  Residual                     0.00 USD
  TOTAL                   33,676.06 USD

reconciliation_check: reconciled=True residual=2.183e-11 tolerance=1e-06
attribution total_pnl : 33,676.06
stressed PV - base PV : 33,676.06


### Per-position drill-down

`by_position_json()` returns the same factor buckets keyed by `position_id`, in Rust `IndexMap` order. Note that each position reports in its **own** currency — `P-EUR` is in EUR. The base-currency translation of that EUR PV is exactly what shows up as portfolio-level `fx_translation_pnl`, which is why no position carries an `fx_translation_pnl` of its own.

In [4]:
by_position = json.loads(attribution.by_position_json())

header = f"{'position':<8} {'ccy':<4} {'rates':>13} {'credit':>13} {'cross':>10} {'total':>13}"
print(header)
print("-" * len(header))
for position_id, entry in by_position.items():
    money = entry["total_pnl"]
    print(
        f"{position_id:<8} {money['currency']:<4} "
        f"{float(entry['rates_curves_pnl']['amount']):>13,.2f} "
        f"{float(entry['credit_curves_pnl']['amount']):>13,.2f} "
        f"{float(entry['cross_factor_pnl']['amount']):>10,.2f} "
        f"{float(money['amount']):>13,.2f}"
    )

position ccy          rates        credit      cross         total
------------------------------------------------------------------
P-USD    USD      -3,375.02          0.00       0.00     -3,375.02
P-EUR    EUR      -1,351.76          0.00       0.00     -1,351.76
P-CDS    USD        -831.54     51,777.19     556.03     51,501.68


## Pedagogical cross-check: naive single-factor differencing

> **This cell is a teaching contrast, not the production path.** The engine's decomposition above is authoritative. Here we deliberately hand-roll the naive alternative — apply each single-factor scenario to the *base* market on its own via `apply_scenario_and_revalue`, difference the PVs, and treat the leftover as an interaction term — so you can see how it differs.

The two approaches isolate factors from opposite ends:

- **Naive (forward isolation)**: shock one factor away from T₀ while everything else stays at T₀.
- **`Parallel` (backward isolation)**: restore one factor to T₀ while everything else sits at T₁.

Both must reconcile to the *same* total P&L, but the individual buckets and the sign of the interaction term differ. The FX bucket happens to agree exactly, because translating an EUR PV into USD is separable from the curve shocks; the rates and credit buckets do not agree, because a wider hazard curve changes the very cashflows the discount shock acts on.

In [5]:
from finstack_quant.portfolio import apply_scenario_and_revalue

# CROSS-CHECK ONLY — the authoritative decomposition is `attribution` above.
naive = {}
for label, scenario in (("Rates", rates_scn), ("Credit", credit_scn), ("FX", fx_scn)):
    valuation_json, report_json = apply_scenario_and_revalue(portfolio, scenario, market_json)
    stressed = PortfolioValuation.from_json(valuation_json).total_value
    naive[label] = stressed - base_total
    print(f"{label:<7} ops={json.loads(report_json)['operations_applied']}  dP&L={naive[label]:,.2f}")

naive["Cross-factor"] = (stressed_total - base_total) - sum(naive.values())

engine = {
    "Rates": attribution.rates_curves_pnl.amount,
    "Credit": attribution.credit_curves_pnl.amount,
    # A foreign-currency deposit has no FX-instrument leg, so its FX effect is pure translation.
    "FX": attribution.fx_pnl.amount + attribution.fx_translation_pnl.amount,
    "Cross-factor": attribution.cross_factor_pnl.amount,
}

header = f"\n{'bucket':<14} {'Parallel engine':>18} {'naive singles':>16} {'difference':>13}"
print(header)
print("-" * (len(header) - 1))
for label in ("Rates", "Credit", "FX", "Cross-factor"):
    print(f"{label:<14} {engine[label]:>18,.2f} {naive[label]:>16,.2f} {engine[label] - naive[label]:>13,.2f}")
print("-" * (len(header) - 1))
print(
    f"{'TOTAL':<14} {attribution.total_pnl.amount:>18,.2f} "
    f"{sum(naive.values()):>16,.2f} {attribution.total_pnl.amount - sum(naive.values()):>13,.2f}"
)
print("\nBoth columns reconcile to the same total; only the split differs.")

Rates   ops=2  dP&L=-5,110.43


Credit  ops=1  dP&L=52,327.64


FX      ops=1  dP&L=-13,034.49

bucket            Parallel engine    naive singles    difference
----------------------------------------------------------------
Rates                   -5,622.67        -5,110.43       -512.23
Credit                  51,777.19        52,327.64       -550.45
FX                     -13,034.49       -13,034.49         -0.00
Cross-factor               556.03          -506.65      1,062.68
----------------------------------------------------------------
TOTAL                   33,676.06        33,676.06         -0.00

Both columns reconcile to the same total; only the split differs.


## Takeaways

- **`attribute_portfolio_pnl(portfolio, market_t0, market_t1, as_of_t0, as_of_t1, method)`** is the canonical factor decomposition. Never sum apply-and-diff deltas in Python and call the leftover an interaction term — `cross_factor_pnl` *is* that term, computed by the engine.
- **`reconciliation_check(tolerance)`** returns `{'total_residual', 'is_reconciled', 'tolerance'}` so the engine, not the notebook, proves the buckets add back to `total_pnl`.
- **Pick the method deliberately**: `Parallel` for exact repricing with an explicit interaction bucket, `Waterfall` when the report must be additive, `MetricsBased`/`Taylor` when speed or risk-report language matters more than exactness under large shocks.
- **`compose_scenarios`** merges per-factor `ScenarioSpec`s using engine `priority` ordering — do not concatenate `operations` arrays by hand.
- **Typed `OperationSpec` builders** (`curve_parallel_bp`, `market_fx_pct`, …) replace raw-JSON operation dicts and validate at construction.
- **`PortfolioValuation.from_json(...).total_value`** is the typed accessor for base-currency PV; no `total_base_ccy["amount"]` digging.
- **FX shows up in two places**: `fx_pnl` for FX-sensitive instruments, `fx_translation_pnl` for translating foreign-currency PVs into base currency. A plain EUR deposit only touches the latter.
- **FX quotes must exist** in the `MarketContext` before serialization for `market_fx_pct` to bite.